# K Nearest Neighbours (XAI Analysis)


**Project:** Pokémon Type Prediction from Sprite Colors (XAI)

**Team:** Grifo Amarillo

**Model:** KNN 

**Task:** Predict Pokémon primary type from sprite color features (17 classes, 42 features)

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from PIL import Image
import shap
import lime
import lime.lime_tabular
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

cwd = Path.cwd()
if (cwd / "src").is_dir():
    src_path = cwd / "src"
elif (cwd.parent / "src").is_dir():
    src_path = cwd.parent / "src"
else:
    raise FileNotFoundError("Could not find the 'src' directory. Check your working directory.")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import common

plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid")

## 1. Data loading and preparation

We start by loading the data and doing a train/test split based on the `common.py` module.

In [ ]:
df = common.load_data()
type_to_int, int_to_type = common.get_label_mapping()
feature_cols = common.FEATURE_COLS_ALL

X_train, X_test, y_train, y_test, split_idx = common.get_train_test_split(df)
scaler = common.get_scaler(X_train)

# Scaling is CRITICAL for KNN (distance-based)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=feature_cols, index=X_test.index)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")